# ContextNet + MultiNicheNet Workflow

This tutorial demonstrates how to visualize **multi-condition** cell-cell communication data from MultiNicheNet using ccflowR Sankey diagrams.

### What You'll Learn

1. Load MultiNicheNet output with group information
2. Define cell type and group colors
3. Create group-colored Sankey diagrams
4. Compare individual conditions side-by-side
5. Customize and filter for publication figures
6. Export figures

For detailed parameter reference (alpha, labels, fonts, layout, interactive filters, export), see the **Parameter Reference Guide**.

### When to Use MultiNicheNet

| Scenario | Use |
|----------|-----|
| Single condition or timepoint | **NicheNet** |
| Comparing Control vs Treatment | **MultiNicheNet** |
| Multiple disease states | **MultiNicheNet** |
| Developmental stages | **MultiNicheNet** |
| Spatial zones | **MultiNicheNet** |

MultiNicheNet extends NicheNet by adding **differential analysis** between conditions, providing per-condition prioritization scores, log fold-change values, and group-stratified communication networks.

## Data Requirements

### Required Columns
| Column | Type | Description |
|--------|------|-------------|
| `sender` | character | Sending cell type |
| `receiver` | character | Receiving cell type |
| `ligand` | character | Ligand gene symbol |
| `receptor` | character | Receptor gene symbol |
| `group` | character | Condition label (e.g., "Control", "Treatment") |

### Optional Columns (for Flow-Conserving Mode)
| Column | Type | Description |
|--------|------|-------------|
| `prioritization_score` | numeric | MultiNicheNet composite score |
| `scaled_lfc_ligand` | numeric | Differential ligand expression |
| `scaled_lfc_receptor` | numeric | Differential receptor expression |

ccflowR automatically maps these MultiNicheNet columns to its internal names via `prepare_sankey_data()`.

### Loading Your Own Data

    # Load your MultiNicheNet output file
    data <- read_multinichenet("path/to/multinichenet_output.csv")
    table(data$group)

In [ ]:
library(ccflowR)

## Step 1: Load Data

In [ ]:
# Use example data for demonstration
ex <- example_multinichenet()
data <- ex$data

# Inspect the data structure
head(data)

# View the groups
table(data$group)

In [ ]:
# Get a summary of the communication data
summarize_data(data)

You should see 27 interactions across 3 conditions with different network densities: Control has 6 core interactions, Treatment has 9 (adding immune-activation pathways like TGFB1 from Fibroblast and IL6 from Macrophage), and Disease has 12 (adding inflammatory pathways like CCL2 and expanded IFNG/IL6 signaling to B_cell). This mirrors real biology where disease conditions activate additional signaling. The data includes `prioritization_score`, `scaled_lfc_ligand`, and `scaled_lfc_receptor` columns, which enable flow-conserving mode with expression-weighted edge thickness.

## Step 2: Define Colors

**Why two color sets?** MultiNicheNet analysis requires two separate color mappings. Cell type colors identify senders and receivers. Group colors map to experimental conditions so flows can be colored by condition, making it possible to see at a glance which pathways are active in which condition.

In [ ]:
# Cell type colors (from example data)
celltype_colors <- ex$colors
print(celltype_colors)

# Group/condition colors (from example data)
group_colors <- ex$group_colors
print(group_colors)

# Verify all groups have colors
groups_in_data <- unique(data$group)
missing <- setdiff(groups_in_data, names(group_colors))
if (length(missing) > 0) {
  cat("Warning: No colors defined for:", paste(missing, collapse = ", "), "\n")
} else {
  cat("All groups have colors assigned\n")
}

## Step 3: Create a Group-Colored Sankey

**Why color by group?** This is the defining feature of MultiNicheNet visualization. Coloring flows by condition lets you see which signaling pathways are shared across conditions and which are condition-specific.

Two functions work together: `sankey_colors(by = "condition")` tells ccflowR to color flows by condition, and `sankey_group_colors()` provides the color mapping.

In [ ]:
p <- sankey_plot(data, celltype_colors, group_col = "group") +
  sankey_colors(by = "condition") +
  sankey_group_colors(group_colors) +
  sankey_alpha(0.6) +
  sankey_title("Cell Communication by Condition")
p

**Reading the plot:**
- Each flow is colored by its experimental condition
- Where flows for the same interaction overlap, you can see which conditions share that pathway
- Flow thickness reflects the prioritization score for each condition — a thicker Treatment flow means that interaction is more active in Treatment
- Ligand-receptor pairs present in all three conditions appear as stacked colored flows at the same position

## Step 4: Compare Individual Conditions

The combined group-colored plot shows all conditions at once but can become dense. Per-condition plots provide cleaner views for direct side-by-side comparison. Use the same `sankey_order()` across all panels to keep node positions consistent.

In [ ]:
# Filter to Control only
control_data <- data[data$group == "Control", ]

p <- sankey_plot(control_data, celltype_colors) +
  sankey_colors(by = "sender") +
  sankey_alpha(0.7) +
  sankey_title("Control Condition")
p

In [ ]:
# Filter to Disease only
disease_data <- data[data$group == "Disease", ]

p <- sankey_plot(disease_data, celltype_colors) +
  sankey_colors(by = "sender") +
  sankey_alpha(0.7) +
  sankey_title("Disease Condition")
p

Compare the two plots: Control has only 6 interactions, while Disease adds inflammatory pathways (CCL2, expanded IFNG/IL6 signaling to B_cell). Macrophage sends 2 signals in Control but 4 in Disease, and shared pathways like TGFB1 show progressively stronger signals.

## Step 5: Customize Your Figure

ccflowR customizations work the same way with multi-condition data. See the **Parameter Reference Guide** for exhaustive visual examples of every parameter.

### Highlighting Pathways

Combine highlighting with condition coloring to answer questions like "How does Macrophage signaling differ across conditions?"

In [ ]:
# Highlight Macrophage signaling across all conditions
p <- sankey_plot(data, celltype_colors, group_col = "group") +
  sankey_colors(by = "condition") +
  sankey_group_colors(group_colors) +
  sankey_highlight(senders = "Macrophage", dim_alpha = 0.1) +
  sankey_alpha(0.7) +
  sankey_title("Macrophage Signaling by Condition")
p

### Custom Node Ordering

Keep node positions consistent across all panels for direct visual comparison.

In [ ]:
p <- sankey_plot(data, celltype_colors, group_col = "group") +
  sankey_colors(by = "condition") +
  sankey_group_colors(group_colors) +
  sankey_order(
    senders = c("Macrophage", "T_cell", "B_cell", "Fibroblast", "Endothelial"),
    receivers = c("T_cell", "B_cell", "Macrophage", "Fibroblast", "Endothelial")
  ) +
  sankey_alpha(0.6) +
  sankey_title("Custom Node Order")
p

## Step 6: Filter Top Interactions

MultiNicheNet data often contains many interactions. Use `filter_top_pathways()` with `per_group = TRUE` to get the top N from each condition, ensuring balanced representation across groups.

In [ ]:
# Filter to top 5 per condition — ensures balanced representation
top_per_group <- filter_top_pathways(data, n = 5, per_group = TRUE, group_col = "group")
cat("Interactions per group after filtering:\n")
print(table(top_per_group$group))

p <- sankey_plot(top_per_group, celltype_colors, group_col = "group") +
  sankey_colors(by = "condition") +
  sankey_group_colors(group_colors) +
  sankey_alpha(0.6) +
  sankey_title("Top 5 per Condition (Balanced)")
p

## Step 7: Export

Save your figure as HTML (interactive) or static formats (PNG, PDF, SVG). See the **Parameter Reference Guide** for all export options and format details.

In [ ]:
# Build a publication-ready figure
p <- sankey_plot(data, celltype_colors, group_col = "group") +
  sankey_colors(by = "condition") +
  sankey_group_colors(group_colors) +
  sankey_highlight(senders = "Macrophage") +
  sankey_alpha(0.6) +
  sankey_title("Macrophage Signaling by Condition")

# Save as interactive HTML (always works, no dependencies)
sankey_save(p, "example_output.html")
cat("Saved HTML\n")

# Save as static PNG (requires kaleido or webshot2)
# sankey_save(p, "example_output.png", width = 700, height = 900)

# Clean up
if (file.exists("example_output.html")) file.remove("example_output.html")

## Complete Workflow

    library(ccflowR)

    # 1. Load your MultiNicheNet output
    data <- read_multinichenet("path/to/multinichenet_output.csv")
    print(table(data$group))

    # 2. Define colors
    celltype_colors <- c(
      "B Cell" = "#17BECF", "T Cell" = "#006837",
      "Myeloid" = "#FF7F0E", "Fibroblast" = "#9C46D3",
      "Epithelial" = "#1F77B4", "Endothelial" = "#E89FBD"
    )
    group_colors <- c(
      "Control" = "#ADD8E6", "Treatment" = "#F08080", "Disease" = "#90EE90"
    )

    # 3. Create comparison figure
    p <- sankey_plot(data, celltype_colors, group_col = "group") +
      sankey_colors(by = "condition") +
      sankey_group_colors(group_colors) +
      sankey_alpha(0.6) +
      sankey_title("Condition Comparison")

    # 4. Save (see Parameter Reference Guide for all export options)
    sankey_save(p, "Figure2_comparison.html")
    sankey_save(p, "Figure2_comparison.png", width = 700, height = 900)

## Troubleshooting

| Issue | Solution |
|-------|----------|
| "group column not found" | Ensure your data has a `group` column, or specify `group_col = "your_column_name"` |
| Groups not colored correctly | Check that `group_colors` names match exactly the values in your `group` column |
| Colors look wrong in condition mode | Make sure you're using `sankey_colors(by = "condition")` AND `sankey_group_colors()` |
| Too many overlapping flows | Use lower alpha (e.g., `sankey_alpha(0.4)`) or filter to fewer interactions with `filter_top_pathways(per_group = TRUE)` |

For general troubleshooting (export, labels, layout), see the **Parameter Reference Guide**.

## Next Steps
- **Single condition analysis**: See the [NicheNet tutorial](tutorial-contextnet-nichenet.ipynb) for detailed single-condition workflow
- **All parameters**: See the [Parameter Reference Guide](tutorial-parameter-guide.ipynb) for visual examples of every customization option